# `evm-intervene`: Counterfactual Intervention Benchmark & GPU Scaling Sweep

**Physical Reasoning Working Group — Answering [`physicalreasoning/forward` Issue #5](https://github.com/physicalreasoning/forward/issues/5)**

This notebook runs the complete **`evm-intervene`** pipeline on a single Google Colab **T4 or L4 GPU** in **~5 minutes** (`< 0.5` Colab compute units):
1. **Zero-Auth Mainnet Data Fetch (`~35s`):** Downloads `31,588` real Ethereum Mainnet Uniswap v3 (`USDC/WETH 0.05%`) swaps, `180` historical `TickLens` concentrated-liquidity boundary maps, and `185` real Aave v3 near-threshold borrower positions from the August 5, 2024 Yen-Carry Crash.
2. **Gate G0 — Exact Simulator Verification (`~8s`):** Verifies the deterministic two-stage counterfactual simulator $\mathcal{F} = \mathcal{F}_{\text{cas}} \circ \mathcal{F}_{\text{AMM}}$ against `31,283` real Mainnet swaps (`4.44e-12 bps` median error).
3. **Mainnet Intervention Benchmark (`~70s`):** Evaluates 4 world-model families (`mlp_direct`, `seq_transformer`, `latent_jepa`, `conservation_wm`) + 5 baselines across 5 seeds on **Regime I (In-Tick)**, **Regime II (Tick-Cliff Crossing)**, and **Regime III (Aave v3 Liquidation Cascade)**.
4. **GPU Scaling Law Sweep (`150k` $\to$ `25M` parameters + `TabPFN v2`, `~3 min`):** Tests whether scaling unstructured neural world models by `160x` in parameters (`150k` $\to$ `25M` in `bf16`/`fp16`) closes the counterfactual intervention gap against the `2,629`-parameter conservation-law world model.

In [ ]:
# 1. Check GPU & Unpack Self-Contained Repository + Pre-Verified Mainnet Data (~5 seconds)
import base64, io, os, sys, tarfile, time, torch

assert torch.cuda.is_available(), 'Please enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'[gpu] Detected {gpu_name} ({vram_gb:.1f} GB VRAM)\n')
REPO_TAR_XZ_B64 = '...'
print('[unpack] Decompressing repository and 31,588 Mainnet swaps...')
raw_bytes = base64.b64decode(REPO_TAR_XZ_B64)
with tarfile.open(fileobj=io.BytesIO(raw_bytes), mode='r:xz') as tar:
    members = tar.getmembers()
    for i, member in enumerate(members, 1):
        try:
            tar.extract(member, '.', filter='data')
        except TypeError:
            tar.extract(member, '.')
        pct = (i / len(members)) * 100
        bar = '=' * int(pct // 4) + '-' * (25 - int(pct // 4))
        sys.stdout.write(f'\r[unpack] [{bar}] {pct:5.1f}% ({i}/{len(members)} files): {member.name[:35]:<35}')
        sys.stdout.flush()
print('\n[unpack] Complete! Code, swaps, checkpoints, and Aave positions ready in /content\n')

print('[install] Installing foundation dependencies (transformers, bitsandbytes, tabpfn, chronos)...')
!pip install -q tabpfn chronos-forecasting transformers accelerate bitsandbytes
print('[install] All dependencies ready! Proceed to Cell 2.\n')


In [ ]:
# 2. Verify Deterministic Simulator on 31,283 Real Mainnet Swaps (Gate G0) (~8 seconds)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
if not os.path.exists('data/mainnet_swaps.json'):
    subprocess.run([sys.executable, 'scripts/download_data.py'], env=env, check=True)
subprocess.run([sys.executable, 'scripts/verify_simulator.py'], env=env, check=True)


In [ ]:
# 3. Run Positive Control & Inspect Pre-Computed 5-Seed Mainnet Benchmark (~15 seconds)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, 'scripts/run_positive_control.py'], env=env, check=True)
if not os.path.exists('results/mainnet_intervention_results.json'):
    subprocess.run([sys.executable, 'scripts/run_experiment.py', '--device', 'cuda'], env=env, check=True)
subprocess.run([sys.executable, 'scripts/make_figures.py'], env=env, check=True)


In [ ]:
# 4. Run Scaling Sweep: 150k -> 1.5M -> 8M -> 85M -> 7 BILLION (Qwen2.5-7B 4-bit) (~10-12 mins on T4)
import base64, io, os, subprocess, sys, tarfile
if 'REPO_TAR_XZ_B64' in globals():
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(REPO_TAR_XZ_B64)), mode='r:xz') as tar:
        tar.extractall('.')
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run(
    [sys.executable, '-u', 'scripts/run_scaling_sweep.py', '--device', 'cuda', '--epochs', '45', '--include-7b', '--include-tabpfn', '--include-chronos'],
    env=env,
    check=True,
)


In [ ]:
# 5. Visualize Scaling Sweep Results: Parameter Count (2.5k -> 7B Foundation) vs. Cliff Error
import json, math, os, subprocess, sys

if not os.path.exists('results/scaling_sweep_results.json'):
    print('Running scaling sweep first...')
    env = {**os.environ, 'PYTHONPATH': '.:src'}
    subprocess.run([sys.executable, '-u', 'scripts/run_scaling_sweep.py', '--device', 'cuda', '--epochs', '45', '--include-7b', '--include-tabpfn', '--include-chronos'], env=env, check=True)

with open('results/scaling_sweep_results.json') as f:
    sweep = json.load(f)

os.makedirs('assets', exist_ok=True)
svg_lines = ['<svg xmlns="http://www.w3.org/2000/svg" width="980" height="440" viewBox="0 0 980 440">',
             '<rect width="100%" height="100%" fill="#f8fafc"/>',
             '<text x="32" y="36" font-family="sans-serif" font-size="18" font-weight="bold" fill="#0f172a">Complete Scaling Sweep: 2.5k to 7.6 Billion Parameters vs. Counterfactual Cliff Error</text>']
y_row = 70
anchor = sweep['anchor_conservation_wm']
svg_lines.append(f'<text x="32" y="{y_row}" font-family="monospace" font-size="13" fill="#059669">[Structure-Preserving Anchor] conservation_wm ({anchor["params"]:,d} params): obs_val={anchor["obs_val_std_mse_mean"]:.4f} | cliff_rmse={anchor["cliff_rmse_bps_mean"]:.2f} bps</text>')
for t_name, t_data in sweep['scaling_tiers'].items():
    for fam, fd in t_data.items():
        y_row += 20
        svg_lines.append(f'<text x="32" y="{y_row}" font-family="monospace" font-size="12" fill="#1e293b">{t_name:8s} | {fam:16s} | params={fd["params"]:11,d} | obs_val={fd["obs_val_std_mse_mean"]:.4f} | cliff_rmse={fd["cliff_rmse_bps_mean"]:.2f} bps</text>')
if sweep.get('foundation_7b_model'):
    f7 = sweep['foundation_7b_model']
    y_row += 22
    svg_lines.append(f'<text x="32" y="{y_row}" font-family="monospace" font-size="13" font-weight="bold" fill="#dc2626">[Frontier 7B Foundation] {f7["family"]} ({f7["params"]:,d} params): obs_val={f7["obs_val_std_mse"]:.4f} | cliff_rmse={f7["cliff_rmse_bps"]:.2f} bps</text>')
svg_lines.append('</svg>')
with open('assets/scaling_sweep_colab.svg', 'w', encoding='utf-8') as sf:
    sf.write('\n'.join(svg_lines))

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), dpi=130)
    colors = {'mlp_direct': '#2563eb', 'seq_transformer': '#9333ea', 'latent_jepa': '#d97706'}
    labels = {'mlp_direct': 'Direct MLP (150k -> 85M)', 'seq_transformer': 'Cross-Attn Transformer (150k -> 85M)', 'latent_jepa': 'Latent JEPA VICReg (150k -> 85M)'}
    for fam in ['mlp_direct', 'seq_transformer', 'latent_jepa']:
        params, obs_mse, cliff_bps = [], [], []
        for tier_name, tier_data in sweep['scaling_tiers'].items():
            if fam in tier_data:
                params.append(tier_data[fam]['params'])
                obs_mse.append(tier_data[fam]['obs_val_std_mse_mean'])
                cliff_bps.append(tier_data[fam]['cliff_rmse_bps_mean'])
        axes[0].plot(params, obs_mse, marker='o', linewidth=2.2, color=colors[fam], label=labels[fam])
        axes[1].plot(params, cliff_bps, marker='o', linewidth=2.2, color=colors[fam], label=labels[fam])
    axes[0].axhline(anchor['obs_val_std_mse_mean'], color='#059669', linestyle='--', linewidth=2.2, label=f"Conservation WM ({anchor['params']:,} params)")
    axes[1].axhline(anchor['cliff_rmse_bps_mean'], color='#059669', linestyle='--', linewidth=2.2, label=f"Conservation WM ({anchor['params']:,} params)")
    if sweep.get('foundation_7b_model'):
        f7 = sweep['foundation_7b_model']
        axes[0].scatter([f7['params']], [f7['obs_val_std_mse']], color='#dc2626', marker='*', s=220, zorder=6, label='Qwen2.5-7B (7.6B Foundation)')
        axes[1].scatter([f7['params']], [f7['cliff_rmse_bps']], color='#dc2626', marker='*', s=220, zorder=6, label='Qwen2.5-7B (7.6B Foundation)')
    if sweep.get('tabpfn_v2_baseline'):
        tp = sweep['tabpfn_v2_baseline']
        axes[0].scatter([tp['params']], [tp['obs_val_std_mse']], color='#ea580c', s=95, zorder=5, label='TabPFN v2 (11M Pretrained)')
        axes[1].scatter([tp['params']], [tp['cliff_rmse_bps']], color='#ea580c', s=95, zorder=5, label='TabPFN v2 (11M Pretrained)')
    if sweep.get('chronos_bolt_small_baseline'):
        cb = sweep['chronos_bolt_small_baseline']
        axes[0].scatter([cb['params']], [cb['obs_val_std_mse']], color='#0891b2', marker='D', s=90, zorder=5, label='Chronos-Bolt-Small (48M Probe)')
        axes[1].scatter([cb['params']], [cb['cliff_rmse_bps']], color='#0891b2', marker='D', s=90, zorder=5, label='Chronos-Bolt-Small (48M Probe)')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Model Parameters (log scale: 2.5k -> 7.6B)')
    axes[0].set_ylabel('Observational Validation Loss (Std MSE)')
    axes[0].set_title('A. Observational Loss Decreases with Scale')
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(fontsize=7.5)
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Model Parameters (log scale: 2.5k -> 7.6B)')
    axes[1].set_ylabel('Counterfactual Tick-Cliff Price RMSE (bps)')
    axes[1].set_title('B. 7.6B Foundation Model Still Collapses on Cliffs')
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(fontsize=7.5)
    plt.tight_layout()
    plt.savefig('assets/scaling_sweep_colab.png', dpi=150)
    plt.show()
except ImportError:
    print('Saved publication SVG to assets/scaling_sweep_colab.svg')

print('Dataset Summary:', json.dumps(sweep['dataset_summary'], indent=2))
if 'foundation_7b_model' in sweep and sweep['foundation_7b_model']:
    print('7B Foundation Model Result:', json.dumps(sweep['foundation_7b_model'], indent=2))
print('Global Spearman rho(Obs Loss, Cliff RMSE):', json.dumps(sweep['global_spearman_obs_vs_cliff_rmse'], indent=2))
